<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 12 - Chart Critique Practice

*(Adapted from Part II - Visualization II)*

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB12_b.ipynb")

# Visualizing Two Distributions: Stacked Histogram → Age Pyramid


When comparing **two distributions side by side**, the chart you choose changes what the reader can actually see.  
This notebook covers one specific upgrade: replacing a **stacked histogram** (hard to compare) with a  
**diverging bar / population pyramid** (easy to compare directly).

---
**Run the setup cell first.**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

np.random.seed(42)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
})



<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## The Problem with Stacked Histograms

When two groups share the same x-axis (e.g., age), a stacked histogram places one group  
on top of the other. This means:

- Group A's bars start at 0 — easy to read   
- Group B's bars start wherever Group A ends — **impossible to judge their shape**

The reader cannot compare the two distributions; they can only see the total.

> *Wilke (2019): "Stacked histograms are not recommended for comparing distributions."*


### Before — Stacked Histogram

**What the code does:**
- Generates two synthetic age distributions (Group A: younger skew, Group B: older skew)
- Bins ages into 2-year intervals using `pd.cut()`
- Plots both groups as a stacked bar chart with `bottom=` to stack Group B on top of Group A
- Problem: Group B's shape is completely distorted by the varying baseline


In [ ]:
# --- Generate synthetic age data ---
group_a = pd.Series(np.random.normal(loc=25, scale=10, size=500).clip(0, 75))
group_b = pd.Series(np.random.normal(loc=35, scale=12, size=500).clip(0, 75))

# Bin into 2-year age groups
bins = range(0, 76, 2)
labels = [f"{i}" for i in range(0, 74, 2)]

counts_a = pd.cut(group_a, bins=bins, labels=labels).value_counts().sort_index()
counts_b = pd.cut(group_b, bins=bins, labels=labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(labels))

# Group A bars from zero
ax.bar(x, counts_a, color='#4a90d9', label='Group A', width=0.9)

# Group B bars stacked ON TOP of Group A — X distorts Group B's shape
ax.bar(x, counts_b, bottom=counts_a, color='#f0a500', label='Group B', width=0.9)

ax.set_xticks(x[::5])
ax.set_xticklabels(labels[::5])
ax.set_xlabel('age (years)')
ax.set_ylabel('count')
ax.set_title('Visualizing Multiple Distributions Together')
ax.legend(title='group')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


**What's wrong here?**  
- Group A (blue) is readable — it starts at zero  
- Group B (orange) floats at varying heights — its shape is impossible to judge  
- You can't answer: *"Are 30-year-olds more common in Group B than Group A?"*


---
### After: Population Pyramid (Diverging Bar Chart)

**The fix:** Mirror the two groups on opposite sides of a shared center axis.  
- Group A extends to the **right** (positive values)  
- Group B extends to the **left** (negative values — we negate them)  
- Both distributions now start from the same baseline: zero  
- The reader can directly compare bar lengths at every age

**What changes in the code:**
- We use `ax.barh()` with **positive values** for Group A and **negated values** for Group B
- `ax.set_xlim()` is set symmetrically so both sides use the same scale  
- `ax.xaxis.set_major_formatter()` uses `abs()` so negative tick labels display as positive numbers  
- Two text labels replace the legend — placed directly above their respective sides


In [ ]:
# Bin into 2-year age groups — same data as before
bins = list(range(0, 76, 2))
bin_labels = list(range(0, 74, 2))

counts_a = pd.cut(group_a, bins=bins, labels=bin_labels).value_counts().sort_index()
counts_b = pd.cut(group_b, bins=bins, labels=bin_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 7))

y = np.array(bin_labels)   # age values on the y-axis

# Group A → right side (positive)
ax.barh(y, counts_a.values, height=1.8, color='#4a90d9', label='Group A')

# Group B → left side (negate the counts so bars extend left)
ax.barh(y, -counts_b.values, height=1.8, color='#f0a500', label='Group B')

# Symmetric x-axis
max_count = max(counts_a.max(), counts_b.max()) + 5
ax.set_xlim(-max_count, max_count)

# Show absolute values on x-axis (hide the minus signs)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: int(abs(x))))

# Direct labels instead of legend
ax.text(-max_count * 0.5, 74, 'group B', ha='center', fontsize=11,
        fontweight='bold', color='#f0a500')
ax.text(max_count * 0.5, 74, 'group A', ha='center', fontsize=11,
        fontweight='bold', color='#4a90d9')

# Center spine only — remove the others
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.axvline(0, color='white', linewidth=1.5)

ax.set_ylabel('age (years)')
ax.set_xlabel('count')
ax.set_title('Visualizing Two Distributions Together')

plt.tight_layout()
plt.show()


**What's better now?**  
- Both groups share the same zero baseline — direct comparison at every age ✅  
- The pyramid shape immediately reveals which group is younger/older  
- Direct labels replace the legend — the reader's eye doesn't have to travel


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🔑 Key Code Concepts

**Negating values to mirror bars left:**

```python
ax.barh(y, -counts_b.values, ...)   # minus sign flips to left side
```

**Hiding the minus signs on the axis:**

```python
def format_tick(x, pos):
    return int(abs(x))

ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_tick))
```
`format_tick` is passed to matplotlib's formatter as the function it calls for every tick.
It takes the raw tick value `x` (which is negative on the left), applies `abs()`, and returns a clean integer.
The `pos` parameter is the tick's position on the axis, which we don't need here but the formatter always passes it.

**Symmetric x limits:**

```python
ax.set_xlim(-max_count, max_count)   # same range on both sides
```
If you skip this, one side will appear larger than the other — misleading the reader.


<!-- BEGIN QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Your Turn

**Scenario:** You are analyzing commute distances (in km) for transit riders vs. car commuters  
in a Bay Area corridor study.

**Steps:**
1. Run the Before cell to see the stacked histogram
2. In the empty cell below, convert it to a population pyramid
3. **Bonus:** Change the color scheme to distinguish transit (green tones) from car (red tones)


In [ ]:
# Practice Dataset A — BEFORE (stacked histogram — fix this!)
np.random.seed(7)
transit = pd.Series(np.random.normal(loc=8, scale=4, size=400).clip(0.5, 40))
car     = pd.Series(np.random.normal(loc=18, scale=7, size=400).clip(0.5, 40))

dist_bins  = list(range(0, 42, 2))
dist_labels = list(range(0, 40, 2))

counts_transit = pd.cut(transit, bins=dist_bins, labels=dist_labels).value_counts().sort_index()
counts_car     = pd.cut(car,     bins=dist_bins, labels=dist_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(dist_labels))

ax.bar(x, counts_transit, color='#27ae60', label='Transit', width=0.9)
ax.bar(x, counts_car, bottom=counts_transit, color='#e74c3c', label='Car', width=0.9)

ax.set_xticks(x[::3])
ax.set_xticklabels(dist_labels[::3])
ax.set_xlabel('Commute Distance (km)')
ax.set_ylabel('Count')
ax.set_title('Commute Distance: Transit vs. Car')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ✏️ YOUR TURN — convert to a population pyramid
# Use the same counts_transit and counts_car computed above

fig, ax = plt.subplots(figsize=(9, 6))

y = np.array(dist_labels)

# Step 1: plot transit on the RIGHT (positive)
# ax.barh(...)

# Step 2: plot car on the LEFT (negate counts_car)
# ax.barh(...)

# Step 3: set symmetric xlim
# max_count = ...
# ax.set_xlim(...)

# Step 4: hide minus signs on x-axis
# ax.xaxis.set_major_formatter(...)

# Step 5: add direct labels instead of legend
# ax.text(...)

ax.set_ylabel('Commute Distance (km)')
ax.set_xlabel('Count')
ax.set_title('Commute Distance: Transit vs. Car')
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Your Turn

**Scenario:** You have building permit application wait times (in days) for  
residential vs. commercial permits in a fictitious city.

**Steps:**
1. Run the Before cell
2. Convert to a population pyramid
3. **Bonus:** The two distributions largely overlap in the 10–30 day range.  
   Add a text annotation on the chart pointing this out — use `ax.text()` or `ax.annotate()`

> *Hint: the annotation should go near the center of the chart at the overlapping range*


In [ ]:
# Practice Dataset B — BEFORE
np.random.seed(99)
residential = pd.Series(np.random.normal(loc=18, scale=6, size=350).clip(1, 60))
commercial  = pd.Series(np.random.normal(loc=28, scale=9, size=350).clip(1, 60))

wait_bins   = list(range(0, 62, 3))
wait_labels = list(range(0, 60, 3))

counts_res = pd.cut(residential, bins=wait_bins, labels=wait_labels).value_counts().sort_index()
counts_com = pd.cut(commercial,  bins=wait_bins, labels=wait_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(wait_labels))

ax.bar(x, counts_res, color='#8e44ad', label='Residential', width=0.9)
ax.bar(x, counts_com, bottom=counts_res, color='#f39c12', label='Commercial', width=0.9)

ax.set_xticks(x[::3])
ax.set_xticklabels(wait_labels[::3])
ax.set_xlabel('Wait Time (days)')
ax.set_ylabel('Count')
ax.set_title('Permit Wait Times: Residential vs. Commercial')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ✏️ YOUR TURN — convert to a population pyramid (+ bonus annotation)

fig, ax = plt.subplots(figsize=(9, 6))

y = np.array(wait_labels)

# --- Add your code here ---


ax.set_ylabel('Wait Time (days)')
ax.set_xlabel('Count')
ax.set_title('Permit Wait Times: Residential vs. Commercial')
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()


<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 💬 Your Take

Which chart type do you prefer for comparing two distributions on the same axis: the **population pyramid**, or the original **stacked histogram**?

Assign your answer to `preferred_chart` as either `"pyramid"` or `"stack"`. There's no wrong answer here -- either is accepted -- but you do need to pick one.

In [ ]:
preferred_chart = ...

In [ ]:
grader.check("q_chart_preference")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 📋 Summary

| | Stacked Histogram | Population Pyramid |
|---|---|---|
| Both groups start at zero | only Group A | yes |
| Can compare shapes directly | no | yes |
| Works for 2 groups | technically | yes best use case |
| Works for 3+ groups | worse | not suitable |
| Key technique | `ax.bar(bottom=)` | `ax.barh()` with negated values |

**When to use a population pyramid:** exactly two groups sharing the same numeric axis  
(age, distance, time, income brackets). Classic use case is demographic comparison.  
For 3+ groups, use **small multiples** (one subplot per group) instead.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Upload your submission zip to **LAB12 - Chart Critique Practice** on [Gradescope](https://www.gradescope.com/).

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)